In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load
old = pd.read_csv(r'C:\Users\Rono\Desktop\GEE\Data\Processed\Kericho_Wards_NDVI_S2_2020_2025.csv')
new = pd.read_csv(r'C:\Users\Rono\Desktop\GEE\Data\Processed\Kericho_Wards_NDVI_S2_v2_2020_2025.csv')
modis = pd.read_csv(r'C:\Users\Rono\Desktop\GEE\Data\Processed\Kericho_Wards_MODIS_2000_2025.csv')

# Standardize columns
old = old.rename(columns={'mean': 'ndvi_old', 'count': 'count_old'})
old['ndvi_old'] = old['ndvi_old'] / 10000
new = new.rename(columns={'NDVI_mean': 'ndvi_new',
                          'valid_count_mean': 'valid_count'})
new['ndvi_new'] = new['ndvi_new'] / 10000

for df in [old, new, modis]:
    df['month'] = pd.to_datetime(df['month'], format='%Y-%m')

modis_recent = modis[modis['month'] >= '2020-01-01'][
    ['ward_name', 'month', 'NDVI_mean']].rename(columns={'NDVI_mean': 'ndvi_modis'})

# Merge all three
merged = (old[['ward_name', 'month', 'ndvi_old', 'count_old']]
          .merge(new[['ward_name', 'month', 'ndvi_new', 'valid_count']],
                 on=['ward_name', 'month'], how='inner')
          .merge(modis_recent, on=['ward_name', 'month'], how='inner'))

# Correlations
r_old = merged['ndvi_old'].corr(merged['ndvi_modis'])
r_new = merged['ndvi_new'].corr(merged['ndvi_modis'])
print(f"Old S2 vs MODIS pooled r: {r_old:.3f}")
print(f"New S2 vs MODIS pooled r: {r_new:.3f}")

per_ward_old = (merged.groupby('ward_name')[['ndvi_old', 'ndvi_modis']]
                .apply(lambda g: g['ndvi_old'].corr(g['ndvi_modis'])))
per_ward_new = (merged.groupby('ward_name')[['ndvi_new', 'ndvi_modis']]
                .apply(lambda g: g['ndvi_new'].corr(g['ndvi_modis'])))
print(f"Old S2 vs MODIS median ward r: {per_ward_old.median():.3f}")
print(f"New S2 vs MODIS median ward r: {per_ward_new.median():.3f}")

# Inspect November 2023 — the worst month before
nov23 = merged[merged['month'] == '2023-11-01']
print(f"\nNov 2023 mean NDVI:")
print(f"  Old S2:  {nov23['ndvi_old'].mean():.3f}")
print(f"  New S2:  {nov23['ndvi_new'].mean():.3f}")
print(f"  MODIS:   {nov23['ndvi_modis'].mean():.3f}")

# Side-by-side time series
cm = merged.groupby('month').agg(
    old=('ndvi_old', 'mean'),
    new=('ndvi_new', 'mean'),
    modis=('ndvi_modis', 'mean')
).reset_index()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(cm['month'], cm['old'], lw=1.4, color='#d6604d',
        alpha=0.7, label='Old S2 (QA60)')
ax.plot(cm['month'], cm['new'], lw=1.8, color='#1a9850',
        label='New S2 (Cloud Score+)')
ax.plot(cm['month'], cm['modis'], lw=1.8, color='#2166ac',
        label='MODIS')
ax.set_ylabel('NDVI')
ax.set_title('Sentinel-2 re-extraction: QA60 vs Cloud Score+')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/s2_reextraction_comparison.png', dpi=150,
            bbox_inches='tight')
plt.show()

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf4 in position 7: invalid continuation byte